# Green and Yellow Apple Detection, Depth-Aware Tracking, and Counting

**Dataset:** `Dataset/DJI_0418.MP4`  
**Classes:** `green_apple` and `yellow_apple`

This is the main project notebook. It implements the requested workflow in the same sequential notebook style: video → 800 frames → verified labels → training-only augmentation → AAM–EBCA–FM–KWConv features → AdaBins relative depth → depth-aware ByteTrack → unique apple count.


## Research objectives and proposed-method mapping

| Research need | Notebook component |
|---|---|
| Partial occlusion, overlapping fruit, multi-scale fruit | **AAM** attribute attention, **EBCA** P3/P4 bidirectional exchange, **FM** contextual focal modulation, **KWConv** adaptive kernel mixture |
| Foreground/background fruit and spatial localization | **AdaBins** pixel-wise monocular depth, attached to every detection as *relative* depth |
| Stable IDs and no duplicate counting | High/low-confidence **depth-aware ByteTrack-style** association and once-only virtual-line counter |

The supplied RGB video has no calibrated depth measurements. AdaBins depth is therefore used only as a **relative association signal**, never reported as a physical distance without camera/depth calibration.


## Data and result folders

- `Dataset/frames/raw/` — exactly 800 temporally uniform annotation frames
- `Dataset/labels/verified/` — human-verified YOLO labels; `0 = green_apple`, `1 = yellow_apple`
- `Dataset/augmentations/` — training-only augmented images and their transformed labels
- `Dataset/yolo/` — temporally disjoint train/validation/test data
- `Results/` — architecture checks, trained weights, metrics, annotated MP4, CSV, and summary JSON

The raw video is never modified. Seed colour proposals are only review aids: green foliage makes automatic green-apple labels unsafe.


In [ ]:
# 1) Project setup — run first after opening the notebook
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
ROOT = next(
    candidate
    for candidate in (cwd, *cwd.parents)
    if (candidate / "Dataset").exists() and (candidate / "Codeing" / "fruit_pipeline.py").exists()
)
CODE_DIR = ROOT / "Codeing"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import fruit_pipeline as fp

paths = fp.project_paths(ROOT)
print("Project root:", ROOT)
print("Classes:", fp.CLASS_NAMES)
print("Raw video:", paths.raw_video)


In [ ]:
# 2) Environment check. Select the "FruitVision (.venv)" kernel before running training/depth cells.
import platform
import importlib.metadata as metadata

packages = ("torch", "ultralytics", "opencv-python", "numpy", "scipy")
versions = {}
for package in packages:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "NOT INSTALLED"

print("Python:", platform.python_version())
print(json.dumps(versions, indent=2))


## A. Video preflight and exactly 800 annotation frames

The video is 1,920 × 1,080 at about 59.94 FPS with 3,825 frames. `13.0` sample FPS gives more than 800 candidates across the clip, then uniform indexing keeps **exactly 800** frames distributed from the beginning to the end of the video.

The existing raw-frame folder currently needs a clean rebuild if it has any count other than 800. Set `REBUILD_FRAME_SET = True` only when you want to replace generated raw frames; it never changes `DJI_0418.MP4`.


In [ ]:
# 3) Mandatory preflight
preflight = fp.video_preflight(paths.raw_video)
preflight


In [ ]:
# 4) Extract exactly 800 temporally uniform frames for annotation.
# Set True once to replace an old/stale generated raw-frame set.
TARGET_FRAMES = 800
REBUILD_FRAME_SET = False

if not preflight["playable"]:
    raise RuntimeError("Video is not playable; do not create labels or augmentations yet.")

current_raw_count = len(list(paths.frames.glob("*.jpg")))
if current_raw_count not in (0, TARGET_FRAMES) and not REBUILD_FRAME_SET:
    raise RuntimeError(
        f"{current_raw_count} raw frames already exist, not {TARGET_FRAMES}. "
        "Set REBUILD_FRAME_SET = True to replace only the generated raw frames."
    )

frame_manifest = fp.extract_uniform_frames(
    paths,
    sample_fps=13.0,
    max_frames=TARGET_FRAMES,
    overwrite=REBUILD_FRAME_SET,
)
raw_count = len(list(paths.frames.glob("*.jpg")))
if len(frame_manifest) != TARGET_FRAMES or raw_count != TARGET_FRAMES:
    raise RuntimeError(f"Expected exactly {TARGET_FRAMES} frames; manifest={len(frame_manifest)}, disk={raw_count}.")
print(f"Extracted {raw_count} frames to {paths.frames}")


In [ ]:
# 5) Audit temporal coverage before annotation.
# The first and last sampled frames should span the whole video.
manifest_path = paths.manifests / "raw_frames.jsonl"
manifest_rows = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line]
print("Sampled frames:", len(manifest_rows))
print("First sample:", manifest_rows[0])
print("Last sample: ", manifest_rows[-1])


## B. Human annotation for green and yellow apples

**Required rule:** annotate every visible apple, including partly occluded and overlapping fruit. Use a tight visible-region box. Assign `green_apple` to class 0 and `yellow_apple` to class 1. For a reviewed frame with no apple, save an empty `.txt` file.

Do **not** train on HSV seed candidates. Leaves share the green range, so seeds are candidate overlays for visual review only. The verified label files are the only ground truth used by the split and training cells.


In [ ]:
# 6) OPTIONAL: create colour seed overlays for visual review only.
# Run after a clean 800-frame extraction. These are NOT ground-truth labels.
CREATE_SEED_PREVIEWS = False
if CREATE_SEED_PREVIEWS:
    seed_summary = fp.create_colour_seed_labels(paths, overwrite=True)
    print(seed_summary)
else:
    print("Skipped. Use LabelImg/OpenCV to create Dataset/labels/verified/ labels.")


In [ ]:
# 7) Open one frame in the local OpenCV annotation helper.
# Draw every apple box, press Enter, then enter g (green), y (yellow), or s (skip) for each box.
# It writes Dataset/labels/verified/<frame-name>.txt in YOLO format.
FRAME_TO_REVIEW = None  # Example: paths.frames / "frame_000000.jpg"

if FRAME_TO_REVIEW is not None:
    verified_boxes = fp.review_frame_with_opencv(paths, FRAME_TO_REVIEW)
    print(f"Saved {len(verified_boxes)} verified boxes.")
else:
    print("Set FRAME_TO_REVIEW to one extracted .jpg path, then run this cell.")


In [ ]:
# 8) Annotation progress. Do not split/train until all 800 frame labels are verified.
raw_frames = sorted(paths.frames.glob("*.jpg"))
verified = {path.stem for path in paths.verified_labels.glob("*.txt")}
missing = [path.name for path in raw_frames if path.stem not in verified]
print(f"Verified: {len(verified & {path.stem for path in raw_frames})}/{len(raw_frames)}")
print("Next frames to label:", missing[:20])


In [ ]:
# 9) Quality-control preview for a verified label file.
# This catches wrong class IDs, malformed YOLO coordinates, and missed overlapping fruit.
import cv2
import matplotlib.pyplot as plt

PREVIEW_FRAME = None  # Example: paths.frames / "frame_000000.jpg"
if PREVIEW_FRAME is not None:
    image = cv2.imread(str(PREVIEW_FRAME))
    boxes = fp.read_yolo_labels(paths.verified_labels / f"{PREVIEW_FRAME.stem}.txt")
    preview = fp.draw_yolo_boxes(image, boxes)
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()
else:
    print("Set PREVIEW_FRAME after its verified label file exists.")


## C. Temporal split and label-safe augmentation

Split original frames first: 70% train, 10% validation, 20% test in temporal order. Only training frames are augmented, preventing adjacent frames or transformed copies from leaking into validation/test.

Augmentations target the problems in the methodology: partial leaf occlusion, low/high illumination, motion blur, and viewpoint reversal. Strong hue augmentation is deliberately excluded because hue distinguishes the green and yellow classes.


In [ ]:
# 10) Build temporal train/validation/test split after all raw frames have verified labels.
BUILD_SPLIT = False
if BUILD_SPLIT:
    if len(missing) != 0:
        raise RuntimeError(f"Label all frames first; {len(missing)} verified label files are missing.")
    split_summary = fp.build_temporal_yolo_dataset(paths, train_fraction=0.70, val_fraction=0.10)
    print(split_summary)
    print("YOLO YAML:", paths.yolo_yaml)
else:
    print("Set BUILD_SPLIT = True after human verification is complete.")


In [ ]:
# 11) Generate training-only augmentations and validate the dataset.
AUGMENT_TRAINING = False
if AUGMENT_TRAINING:
    augmentation_summary = fp.augment_training_split(paths, seed=42, overwrite=False)
    validation_report = fp.validate_yolo_dataset(paths)
    print("Augmentations:", augmentation_summary)
    print(json.dumps(validation_report, indent=2))
else:
    print("Set AUGMENT_TRAINING = True only after the temporal split exists.")


## D. Proposed AAM–EBCA–FM–KWConv feature enhancement

These cells are the runnable implementation of the four modules from the shared methodology:

1. **AAM:** learns colour, shape, and scale attention, then fuses them with learned weights.
2. **EBCA:** exchanges information in both directions between high-resolution P3 and semantic P4 to separate overlapping fruit.
3. **FM:** combines query features and hierarchical local/global context, suppressing background/occluded evidence.
4. **KWConv:** selects a data-dependent mixture of 3×3, 5×5, and 7×7 kernels for small and large apples.

The neck preserves the P3/P4/P5 tensor sizes expected by a detector. The smoke test validates those interfaces before an experiment.


In [ ]:
# 12) PyTorch imports for the proposed feature modules.
import math
import torch
from torch import nn
from torch.nn import functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Feature-module device:", DEVICE)


In [ ]:
# 13) AAM — Attribute Attention Module: colour + shape + scale branches with learned fusion.
class AAM(nn.Module):
    def __init__(self, channels: int, reduction: int = 8) -> None:
        super().__init__()
        hidden = max(1, channels // reduction)
        self.colour = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, hidden, 1), nn.SiLU(), nn.Conv2d(hidden, channels, 1)
        )
        self.shape = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.scale = nn.Sequential(nn.AvgPool2d(3, stride=1, padding=1), nn.Conv2d(channels, channels, 1))
        self.branch_weights = nn.Parameter(torch.zeros(3))

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        colour = self.colour(features).expand_as(features)
        shape = self.shape(features)
        scale = self.scale(features)
        alpha = torch.softmax(self.branch_weights, dim=0)
        attention = torch.sigmoid(alpha[0] * colour + alpha[1] * shape + alpha[2] * scale)
        return features * attention


In [ ]:
# 14) EBCA — efficient bidirectional P3↔P4 cross-attention for overlapping fruit regions.
class EBCA(nn.Module):
    def __init__(self, p3_channels: int, p4_channels: int) -> None:
        super().__init__()
        self.p4_to_p3 = nn.Conv2d(p4_channels, p3_channels, 1, bias=False)
        self.p3_to_p4 = nn.Conv2d(p3_channels, p4_channels, 1, bias=False)
        self.p3_query = nn.Conv2d(p3_channels, p3_channels, 1, bias=False)
        self.p4_query = nn.Conv2d(p4_channels, p4_channels, 1, bias=False)
        self.p3_norm = nn.BatchNorm2d(p3_channels)
        self.p4_norm = nn.BatchNorm2d(p4_channels)

    def forward(self, p3: torch.Tensor, p4: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        p4_up = F.interpolate(self.p4_to_p3(p4), size=p3.shape[-2:], mode="bilinear", align_corners=False)
        p3_gate = torch.sigmoid(self.p3_query(p3) * p4_up / math.sqrt(p3.shape[1]))
        enhanced_p3 = self.p3_norm(p3 + p3_gate * p4_up)
        p3_down = F.adaptive_avg_pool2d(self.p3_to_p4(p3), p4.shape[-2:])
        p4_gate = torch.sigmoid(self.p4_query(p4) * p3_down / math.sqrt(p4.shape[1]))
        enhanced_p4 = self.p4_norm(p4 + p4_gate * p3_down)
        return enhanced_p3, enhanced_p4


In [ ]:
# 15) FM — focal modulation with hierarchical context for partial occlusion.
class FM(nn.Module):
    def __init__(self, channels: int, levels: int = 3) -> None:
        super().__init__()
        self.query = nn.Conv2d(channels, channels, 1)
        self.context = nn.ModuleList(
            [nn.Conv2d(channels, channels, 3 + 2 * level, padding=1 + level, groups=channels) for level in range(levels)]
        )
        self.gates = nn.Conv2d(channels, levels + 1, 1)
        self.modulator = nn.Conv2d(channels, channels, 1)
        self.output = nn.Conv2d(channels, channels, 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        query = self.query(features)
        gates = torch.softmax(self.gates(features), dim=1)
        context, current = 0, features
        for level, layer in enumerate(self.context):
            current = layer(current)
            context = context + gates[:, level : level + 1] * current
        context = context + gates[:, -1:] * F.adaptive_avg_pool2d(features, 1)
        return self.output(query * self.modulator(context))


In [ ]:
# 16) KWConv — kernel-warehouse mixture of 3×3, 5×5, and 7×7 branches for multi-scale fruit.
class KWConv(nn.Module):
    def __init__(self, channels: int, kernels: tuple[int, ...] = (3, 5, 7)) -> None:
        super().__init__()
        self.branches = nn.ModuleList(
            [nn.Conv2d(channels, channels, kernel, padding=kernel // 2, bias=False) for kernel in kernels]
        )
        self.selector = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(channels, len(kernels), 1))
        self.norm = nn.BatchNorm2d(channels)
        self.activation = nn.SiLU()

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        weights = torch.softmax(self.selector(features), dim=1)
        mixed = sum(weights[:, i : i + 1] * branch(features) for i, branch in enumerate(self.branches))
        return self.activation(self.norm(mixed))


In [ ]:
# 17) Full proposed neck: AAM → EBCA → FM → KWConv(P3, P4, P5).
class ProposedEnhancementNeck(nn.Module):
    def __init__(self, p3_channels: int, p4_channels: int, p5_channels: int) -> None:
        super().__init__()
        self.aam = AAM(p4_channels)
        self.ebca = EBCA(p3_channels, p4_channels)
        self.fm = FM(p4_channels)
        self.kw_p3 = KWConv(p3_channels)
        self.kw_p4 = KWConv(p4_channels)
        self.kw_p5 = KWConv(p5_channels)

    def forward(self, p3: torch.Tensor, p4: torch.Tensor, p5: torch.Tensor):
        p4 = self.aam(p4)
        p3, p4 = self.ebca(p3, p4)
        p4 = self.fm(p4)
        return self.kw_p3(p3), self.kw_p4(p4), self.kw_p5(p5)


neck = ProposedEnhancementNeck(96, 192, 384).to(DEVICE).eval()
with torch.inference_mode():
    outputs = neck(
        torch.zeros(1, 96, 80, 80, device=DEVICE),
        torch.zeros(1, 192, 40, 40, device=DEVICE),
        torch.zeros(1, 384, 20, 20, device=DEVICE),
    )
architecture_report = {"device": DEVICE, "output_shapes": [list(item.shape) for item in outputs]}
(paths.results / "architecture_smoke_test.json").write_text(json.dumps(architecture_report, indent=2), encoding="utf-8")
architecture_report


## E. Detector training and validation

The baseline training cell is reproducible now. A stock Ultralytics YOLO checkpoint does not automatically use the notebook feature classes, so report it as **YOLOv8 baseline**. The proposed AAM–EBCA–FM–KWConv neck must be registered in a local Ultralytics model parser and selected in a custom YAML before it can be reported as the proposed detector. The architecture cells above retain a tested, explicit implementation for that integration.


In [ ]:
# 18) Baseline YOLOv8 training. Outputs are written only to Results/training/.
RUN_BASELINE_TRAINING = False
if RUN_BASELINE_TRAINING:
    train_result = fp.train_baseline_yolov8(
        paths,
        model_name="yolov8n.pt",
        epochs=100,
        image_size=1024,
        batch=-1,       # auto-batch; use a smaller integer if memory is limited
        device=0 if torch.cuda.is_available() else "cpu",
    )
    print(train_result)
else:
    print("Set RUN_BASELINE_TRAINING = True after split + augmentation validation.")


In [ ]:
# 19) Validation/test metrics: mAP50-95, precision, recall, and confusion matrix saved under Results/validation/.
RUN_VALIDATION = False
BEST_WEIGHTS = paths.training_results / "yolov8_green_yellow_apple" / "weights" / "best.pt"
if RUN_VALIDATION:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f"No trained weights found: {BEST_WEIGHTS}")
    validation_result = fp.validate_detector(paths, BEST_WEIGHTS, image_size=1024)
    print(validation_result)
else:
    print("Set RUN_VALIDATION = True after successful training.")


## F. AdaBins depth-aware fruit localization

AdaBins estimates a dense depth map from RGB. The central trimmed median inside each detected box is used as the fruit's relative depth value, reducing leaf/background influence near box edges. The map helps distinguish nearby foreground fruit from similar projected background fruit during tracker association.


In [ ]:
# 20) Load the local official AdaBins KITTI checkpoint only when needed.
# The model is CPU-compatible but is resource-intensive; keep it disabled during annotation/training.
USE_ADABINS = False
ADABINS_REPOSITORY = ROOT / "third_party" / "AdaBins"
depth_estimator = None
if USE_ADABINS:
    depth_estimator = fp.AdaBinsDepth(ADABINS_REPOSITORY, dataset="kitti")
    print("AdaBins loaded. Use output as relative orchard depth only.")
else:
    print("AdaBins disabled. Set USE_ADABINS = True for depth-aware tracking.")


In [ ]:
# 21) Optional depth-map inspection on one extracted RGB frame.
DEPTH_PREVIEW_FRAME = None  # Example: paths.frames / "frame_000000.jpg"
if depth_estimator is not None and DEPTH_PREVIEW_FRAME is not None:
    frame = cv2.imread(str(DEPTH_PREVIEW_FRAME))
    depth_map = depth_estimator.predict(frame)
    plt.figure(figsize=(16, 6))
    plt.subplot(1, 2, 1); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("RGB")
    plt.subplot(1, 2, 2); plt.imshow(depth_map, cmap="magma"); plt.colorbar(); plt.axis("off"); plt.title("Relative AdaBins depth")
    plt.show()
else:
    print("Set USE_ADABINS=True and DEPTH_PREVIEW_FRAME to inspect a depth map.")


## G. Depth-aware ByteTrack-style tracking and unique counting

The tracker first associates high-confidence detections, then uses low-confidence detections to recover temporarily missed fruit. Its matching cost combines class consistency, IoU, predicted motion, and relative depth compatibility. A confirmed track is counted once when it crosses the configured line, so the same apple is not counted repeatedly across frames.


In [ ]:
# 22) Final detection + relative depth + two-stage tracking + one-time count.
# Outputs: Results/inference/tracked_apples.mp4, tracks.csv, and summary.json
RUN_FINAL_INFERENCE = False
if RUN_FINAL_INFERENCE:
    if not BEST_WEIGHTS.exists():
        raise FileNotFoundError(f"Train the detector first; missing {BEST_WEIGHTS}")
    frame_width, frame_height = preflight["width"], preflight["height"]
    # Update after inspecting the camera motion: a vertical line at the image centre is a safe starting point.
    counting_line = ((frame_width // 2, 0), (frame_width // 2, frame_height - 1))
    tracking_summary = fp.run_depth_aware_tracking(
        paths,
        weights=BEST_WEIGHTS,
        adabins=depth_estimator,
        low_confidence=0.10,
        image_size=1024,
        counting_line=counting_line,
        count_direction="both",
    )
    print(json.dumps(tracking_summary, indent=2))
else:
    print("Set RUN_FINAL_INFERENCE = True after training. AdaBins remains optional.")


In [ ]:
# 23) Show the reproducible artifacts that have been generated so far.
for folder in (paths.results, paths.training_results, paths.inference_results):
    files = sorted(path.relative_to(ROOT) for path in folder.rglob("*") if path.is_file())
    print(f"\n{folder.relative_to(ROOT)} ({len(files)} files)")
    for path in files[:25]:
        print(" -", path)
    if len(files) > 25:
        print(" - ...")


## Required research result review

Before reporting results, inspect the annotated video and verify:

- mAP50–95, precision, and recall for **each** apple colour;
- performance slices for small, overlapping, and partially occluded apples;
- foreground/background mistakes using the relative depth overlay;
- ID switches and duplicate counts on a manually checked temporal test sequence;
- count MAE against a hand-verified count.

Keep the dataset, labels, augmentation manifests, model configuration, and `Results/` artifacts together locally for reproducibility. Dataset video and generated results stay excluded from the GitHub repository.
